# **Implementing langgraph react agent**

Got it 👍
I’ll **help you resolve the task**, but **in a trainer way** — clear steps, thinking process, and checkpoints — **not dumping full code**.
You’ll know *what to write, why to write it, and where issues usually happen*.

---

# 🎯 Task: Implementing a LangGraph ReAct Agent

Think of this as **solving a problem in phases**, not writing code at once.

---

## 🧩 PHASE 1: Understand What You Are Building

### Your ReAct agent must:

1. Read user input
2. Reason using LLM
3. Decide:

   * “Should I call a tool?”
   * OR “Can I answer now?”
4. If tool is needed:

   * Call tool
   * Observe result
   * Think again
5. Stop only when final answer is ready

👉 This **requires looping**, which is why **LangGraph** is used.

---

## 🧠 PHASE 2: Design the Agent State (Most Important)

Ask yourself:

> “What information must survive between steps?”

Your state should contain:

* Conversation history (`messages`)
* Tool outputs (also messages)
* Optional flags (done / next action)

💡 **Key Rule**

* Use `messages` as a **list**
* Use `operator.add` so messages accumulate

📌 Checkpoint
You should be able to say:

> “My state can store everything the agent thinks, says, and observes.”

---

## 🔁 PHASE 3: Identify the Nodes You Need

A minimal ReAct agent has **3 logical nodes**:

### 1️⃣ Reasoning Node (LLM)

Purpose:

* Think
* Decide next action

This node:

* Reads `messages`
* Produces either:

  * A normal AI message (final answer)
  * OR a `tool_call`

📌 Ask yourself:

> “Does this response request a tool?”

---

### 2️⃣ Tool Node

Purpose:

* Execute the tool requested by LLM

This node:

* Reads tool name + arguments
* Calls correct Python function
* Returns a **ToolMessage**

📌 Important:

> Tool output must go back into `messages`

---

### 3️⃣ Router (Decision Logic)

Purpose:

* Decide next edge

Conditions:

* If LLM output has tool calls → go to Tool Node
* Else → END

📌 This is where **ReAct loop** happens.

---

## 🔀 PHASE 4: Graph Structure (Think Visually)

Your graph should look like this:

```
START
 ↓
Reasoning (LLM)
 ↓
Tool requested?
 ├── Yes → Tool Node → back to Reasoning
 └── No  → END
```

💡 If your graph **does not loop**, it is NOT a ReAct agent.

---

## 🧰 PHASE 5: Tools Thinking (Before Writing Tools)

Before coding tools, answer:

* What actions should my agent perform?
* When should it NOT answer directly?

Examples:

* Calculator → math
* Search → factual lookup
* Custom logic → business rules

📌 Tool rule:

> The LLM must be **aware** of tool descriptions.

---

## ⚠️ PHASE 6: Common Mistakes (Avoid These)

### ❌ Mistake 1: Returning strings instead of message lists

Reducer fields **must return lists**

### ❌ Mistake 2: No conditional edge

Without routing, agent won’t loop

### ❌ Mistake 3: Tool output not added to messages

Then LLM can’t “see” tool results

### ❌ Mistake 4: Treating agent like a chain

ReAct ≠ single pass

---

## 🧪 PHASE 7: How to Debug If It Breaks

Ask in order:

1. Does my state update correctly?
2. Are messages accumulating?
3. Is the LLM actually requesting tools?
4. Does router detect tool calls?
5. Does tool output return as ToolMessage?

👉 Debug **state**, not output.

---

## 🧠 FINAL MENTAL MODEL (MEMORIZE THIS)

```
LLM thinks → Tool acts → LLM observes → LLM decides → END
```

LangGraph = **control flow**
ReAct = **decision logic**
Tools = **capabilities**

---

## ✅ You Are Ready When You Can Answer These

Before coding, confirm:

* What causes the loop?
* What stops the loop?
* Where does reasoning happen?
* Where does action happen?
* Where is state updated?

If you can answer these → **you can implement it** 💪

---

### 🔜 Next (Your Choice)

Say one of these:

* **“Help me design the ReAct state”**
* **“Explain routing logic deeply”**
* **“Walk me through tool + LLM interaction”**
* **“Now help me debug my ReAct code”**

I’ll guide you step-by-step 🚀


In [ ]:
# ===============================
# Imports
# ===============================
from typing import TypedDict, Annotated
import operator

from langgraph.graph import StateGraph
from langgraph.constants import START, END

from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)
from langchain.tools import tool


# ===============================
# 1. Define Tools
# ===============================
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


TOOLS = [add, multiply]
TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}


# ===============================
# 2. Define State
# ===============================
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# ===============================
# 3. Initialize Model
# ===============================
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
).bind_tools(TOOLS)


# ===============================
# 4. Reasoning Node (LLM)
# ===============================
def reasoning_node(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {
        "messages": [response]
    }


# ===============================
# 5. Tool Execution Node
# ===============================
def tool_node(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]

    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        tool_fn = TOOLS_BY_NAME[tool_name]
        result = tool_fn.invoke(tool_args)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_name=tool_name,
                tool_call_id=tool_call["id"]
            )
        )

    return {
        "messages": tool_messages
    }


# ===============================
# 6. Router Logic (ReAct Decision)
# ===============================
def should_call_tool(state: AgentState) -> str:
    last_message = state["messages"][-1]

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tool"
    return "end"


# ===============================
# 7. Build Graph
# ===============================
graph = StateGraph(AgentState)

graph.add_node("reason", reasoning_node)
graph.add_node("tool", tool_node)

graph.add_edge(START, "reason")
graph.add_conditional_edges(
    "reason",
    should_call_tool,
    {
        "tool": "tool",
        "end": END
    }
)
graph.add_edge("tool", "reason")

agent = graph.compile()


# ===============================
# 8. Run the Agent
# ===============================
if __name__ == "__main__":
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content="What is (3 + 5) * 2?")
            ]
        }
    )

    print("\nFinal Answer:\n")
    for msg in result["messages"]:
        print(f"{msg.__class__.__name__}: {msg.content}")



---

# 🔥 ONE-LINE ANSWER FIRST

> **ReAct is NOT a class or function.**
> **ReAct is a BEHAVIOR created by 3 parts working together.**

Those 3 parts are:

1. **Reasoning node (LLM)**
2. **Tool node (Action)**
3. **Looping control (LangGraph)**

Now let’s map this **directly to your code**.

---

# 1️⃣ WHAT “ReAct” REALLY MEANS

**ReAct = Reason + Act**

| Step    | Meaning                    |
| ------- | -------------------------- |
| Reason  | LLM thinks what to do next |
| Act     | LLM calls a tool           |
| Observe | Tool result goes back      |
| Repeat  | LLM reasons again          |

ReAct is a **cycle**, not a function.

---

# 2️⃣ WHERE IS “REASON” IN YOUR CODE?

### 📍 This function is **REASONING**

```python
def reasoning_node(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {
        "messages": [response]
    }
```

### Why this is “Reason”

* The LLM reads conversation history
* It decides:

  * Answer directly ❓
  * OR call a tool 🔧

📌 The **decision** happens inside the model output.

---

# 3️⃣ HOW DOES THE MODEL DECIDE TO “ACT”?

### 📍 This line enables tool reasoning

```python
llm = init_chat_model(...).bind_tools(TOOLS)
```

This tells the LLM:

> “You are allowed to use these tools”

So now the model can output:

```json
{
  "tool_calls": [
    {
      "name": "add",
      "args": {"a": 3, "b": 5}
    }
  ]
}
```

That JSON is the **ACT decision**.

---

# 4️⃣ WHERE IS “ACT” IN YOUR CODE?

### 📍 This function is **ACTION**

```python
def tool_node(state: AgentState) -> AgentState:
```

Inside:

```python
for tool_call in last_message.tool_calls:
    tool_fn = TOOLS_BY_NAME[tool_call["name"]]
    result = tool_fn.invoke(tool_args)
```

### Why this is “Act”

* Executes the real tool
* Gets real output
* No LLM here — pure execution

---

# 5️⃣ WHERE IS “OBSERVE”?

### 📍 This is OBSERVATION

```python
ToolMessage(
    content=str(result),
    tool_name=tool_name,
    tool_call_id=tool_call["id"]
)
```

Why?

* Tool result is converted into a message
* Added back to conversation
* LLM can now “see” the result

---

# 6️⃣ WHERE IS THE “REPEAT” PART?

### 📍 THIS IS THE MOST IMPORTANT PART

```python
graph.add_edge("tool", "reason")
```

This line means:

```
Tool → Reason again
```

Without this edge:
❌ No ReAct
❌ Just a single tool call

This loop is what makes it **ReAct**.

---

# 7️⃣ WHO DECIDES WHETHER TO LOOP OR STOP?

### 📍 Router logic

```python
def should_call_tool(state: AgentState) -> str:
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tool"
    return "end"
```

This means:

* If LLM asks for a tool → continue
* If not → stop

This is **control logic**, not reasoning.

---

# 8️⃣ COMPLETE REACT FLOW (MATCHING CODE)

```
HumanMessage
   ↓
reasoning_node  ←─── Reason
   ↓
tool_calls?
   ├─ YES → tool_node ← Act
   │          ↓
   │     ToolMessage ← Observe
   │          ↓
   └──────────┘
   ↓
Final AIMessage
```

---

# 9️⃣ VERY IMPORTANT REALIZATION 🚨

### ❌ ReAct is NOT:

* A LangGraph class
* A decorator
* A built-in function

### ✅ ReAct IS:

> A **pattern** created by:
>
> * LLM reasoning
> * Tool execution
> * Looping graph

---

# 🔑 ONE SENTENCE TO REMEMBER

> **If your graph loops from LLM → Tool → LLM based on tool_calls, you have a ReAct agent.**

---

# 🧠 INTERVIEW-WORTHY ANSWER

If asked:

> *Where is ReAct in your code?*

Say:

> “The reasoning node is the Reason part, the tool node is the Act part, and LangGraph’s conditional looping creates the ReAct behavior.”

